# FraudGraph — Stage 2: Heterogeneous Graph Construction (v2)Turn the flat 590k-row transaction table into a **HeteroData** graph with fournode types and four edge types. Transaction nodes carry the same full featurematrix used by `xgboost_full`, so Stage 3 isolates graph structure as the onlydelta.**v2 changes (correctness fixes):**- Merchant edges require a present `addr1`. Missing-address rows previously  lumped into per-product "-999" pseudo-merchants — mega-hub nodes carrying no  identity signal. Now they simply get no merchant edge (mirrors how missing  DeviceInfo gets no device edge).- Categorical codes are normalized to (0, 1] by cardinality instead of being  z-scaled. StandardScaler z-scores over arbitrary label codes are semantically  meaningless; unscaled codes (up to ~1800 for DeviceInfo) would blow up the  encoder activations. Bounded normalization fixes both.- `TransactionDT` stored on transaction nodes as `time` for downstream  temporal protocols.

## 0. Setup

In [ ]:
!pip install -q torch_geometric

In [ ]:
import os, pickle, numpy as np, pandas as pd, torch
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import HeteroData

DATA = "/kaggle/input/competitions/ieee-fraud-detection"
OUT = "/kaggle/working"
print("files:", os.listdir(DATA))

## 1. Reload & re-run Stage 1 preprocessing

In [ ]:
txn = pd.read_csv(f"{DATA}/train_transaction.csv")
idn = pd.read_csv(f"{DATA}/train_identity.csv")
df = txn.merge(idn, on="TransactionID", how="left")
df = df.sort_values("TransactionDT").reset_index(drop=True)
cutoff = df["TransactionDT"].quantile(0.80)
is_train = df["TransactionDT"] <= cutoff
print(f"rows: {len(df):,}  cutoff: {int(cutoff):,}  train={is_train.sum():,} test={(~is_train).sum():,}")

In [ ]:
has_addr = df["addr1"].notna()          # capture BEFORE fillna — drives merchant edges
has_id   = df["DeviceInfo"].notna()     # drives device edges

df["card1"] = df["card1"].fillna(-999)
df["addr1"] = df["addr1"].fillna(-999)
df["dt"] = pd.to_datetime(df["TransactionDT"], unit="s")
df["hour"] = ((df["TransactionDT"] // 3600) % 24).astype(int)
df["cm_freq"] = df.groupby(["card1", "ProductCD", "addr1"]).cumcount()

g = df.groupby("card1")["TransactionAmt"]
mean_prev = g.transform(lambda s: s.expanding().mean().shift())
std_prev = g.transform(lambda s: s.expanding().std().shift())
df["amt_zscore"] = ((df["TransactionAmt"] - mean_prev) / std_prev).replace(
    [np.inf, -np.inf], np.nan).fillna(0)

df = df.sort_values(["card1", "dt"])
roll = df.set_index("dt").groupby("card1")["TransactionAmt"]
# NOTE: rolling windows include the current row, so velocity counts include the
# transaction itself (count_1h >= 1). Serving must increment-then-read to match.
df["velocity_1h"] = roll.rolling("1h").count().values
df["velocity_6h"] = roll.rolling("6h").count().values
df["velocity_24h"] = roll.rolling("24h").count().values
df = df.sort_values("TransactionDT").reset_index(drop=True)
is_train = df["TransactionDT"] <= cutoff
has_addr = df["addr1"].notna() & (df["addr1"] != -999)
has_id = df["DeviceInfo"].notna()
print("engineered features ready")
print(f"addr1 present: {has_addr.mean():.1%}   identity present: {has_id.mean():.1%}")

## 2. Node identity keysStable integer ids 0..N-1 per node type. Canonical key semantics (mirrored in`src/features.py` for serving parity):- **card**: float `card1` (missing → -999.0) — every transaction has a card node.- **merchant**: `f"{ProductCD}|{float(addr1)}"`, ONLY when addr1 is present.- **device**: `str(DeviceInfo)`, missing → single `"UNKNOWN"` node.

In [ ]:
DEVICE_UNK = "UNKNOWN"
df["DeviceInfo_key"] = df["DeviceInfo"].fillna(DEVICE_UNK).astype(str)
df["merchant_key"] = df["ProductCD"].astype(str) + "|" + df["addr1"].astype(float).astype(str)

card_ids  = pd.Index(sorted(df["card1"].unique()))
merch_ids = pd.Index(sorted(df.loc[has_addr, "merchant_key"].unique()))
dev_ids   = pd.Index(sorted(df["DeviceInfo_key"].unique()))

card_map  = {v: i for i, v in enumerate(card_ids)}
merch_map = {v: i for i, v in enumerate(merch_ids)}
dev_map   = {v: i for i, v in enumerate(dev_ids)}

df["_card_idx"]  = df["card1"].map(card_map).astype("int64")
df["_merch_idx"] = df["merchant_key"].map(merch_map).fillna(-1).astype("int64")  # -1 = no merchant
df["_dev_idx"]   = df["DeviceInfo_key"].map(dev_map).astype("int64")

print(f"transactions: {len(df):,}")
print(f"cards:        {len(card_map):,}")
print(f"merchants:    {len(merch_map):,}  (rows without merchant: {(df['_merch_idx'] < 0).sum():,})")
print(f"devices:      {len(dev_map):,}  (UNKNOWN idx = {dev_map[DEVICE_UNK]})")

## 3. Transaction node featuresSame column set as `xgboost_full` (parity for the ablation), but scaling istype-aware:- **numerics** → median-impute (train-fit) then StandardScaler (train-fit)- **categoricals** → label codes normalized to (0, 1] by cardinality; missing → 0

In [ ]:
drop_cols = ["TransactionID", "isFraud", "TransactionDT", "dt",
             "DeviceInfo_key", "merchant_key", "_card_idx", "_merch_idx", "_dev_idx"]
cat_cols = [c for c in df.columns if df[c].dtype == "object" and c not in drop_cols]
num_cols = [c for c in df.columns
            if c not in drop_cols + cat_cols and str(df[c].dtype) != "datetime64[ns]"]

cat_maps = {}
Xc = df[cat_cols].copy()
for c in cat_cols:
    m = {v: i for i, v in enumerate(sorted(Xc[c].dropna().unique(), key=str))}
    cat_maps[c] = m
    codes = Xc[c].map(m).fillna(-1).astype("float32")
    Xc[c] = (codes + 1.0) / (len(m) + 1.0)   # missing -> 0, codes -> (0, 1]

Xn = df[num_cols].copy()
med = Xn.loc[is_train].median()
Xn = Xn.fillna(med)
scaler = StandardScaler()
scaler.fit(Xn.loc[is_train].values)
Xn_s = scaler.transform(Xn.values)

X_scaled = np.hstack([Xn_s, Xc.values]).astype(np.float32)
feat_cols = num_cols + cat_cols
y = df["isFraud"].values
print(f"transaction node features: {X_scaled.shape}  ({len(num_cols)} scaled numeric + {len(cat_cols)} normalized categorical)")

## 4. Edge tensorsThree from spec (transaction → outward, skipping rows without the entity) and`velocity_cluster` (transaction → transaction, same card within 5 minutes,strictly earlier → later so message flow is causal).

In [ ]:
tx_idx = np.arange(len(df), dtype=np.int64)

e_uses_card = torch.tensor(np.stack([tx_idx, df["_card_idx"].values]), dtype=torch.long)

am = has_addr.values
e_at_merchant = torch.tensor(
    np.stack([tx_idx[am], df.loc[am, "_merch_idx"].values]), dtype=torch.long)

im = has_id.values
e_on_device = torch.tensor(
    np.stack([tx_idx[im], df.loc[im, "_dev_idx"].values]), dtype=torch.long)

print("uses_card    edges:", e_uses_card.shape[1])
print("at_merchant  edges:", e_at_merchant.shape[1], f"({am.mean():.1%} of tx have addr1)")
print("on_device    edges:", e_on_device.shape[1], f"({im.mean():.1%} of tx have identity)")

In [ ]:
WIN = 300  # seconds
src_all, dst_all = [], []

for card_val, group_pos in df.groupby("card1").indices.items():
    if len(group_pos) < 2:
        continue
    group_pos = np.asarray(group_pos)
    times = df["TransactionDT"].values[group_pos]
    order_local = np.argsort(times, kind="stable")
    times = times[order_local]
    ids = group_pos[order_local]
    upper = np.searchsorted(times, times + WIN, side="right")
    for i in range(len(ids)):
        j0, j1 = i + 1, upper[i]
        if j1 > j0:
            src_all.append(np.full(j1 - j0, ids[i], dtype=np.int64))
            dst_all.append(ids[j0:j1])

if src_all:
    e_velocity_cluster = torch.tensor(
        np.stack([np.concatenate(src_all), np.concatenate(dst_all)]), dtype=torch.long)
else:
    e_velocity_cluster = torch.zeros((2, 0), dtype=torch.long)

print("velocity_cluster edges:", e_velocity_cluster.shape[1])
assert e_velocity_cluster.shape[1] > 0, "velocity_cluster is empty — check window/index alignment"

endpoint_fraud = y[np.unique(e_velocity_cluster.numpy().ravel())].mean()
print(f"Fraud rate overall:                {y.mean():.3%}")
print(f"Fraud rate on velocity_cluster tx: {endpoint_fraud:.3%}  (lift {endpoint_fraud/y.mean():.2f}x)")

## 5. Assemble HeteroData

In [ ]:
data = HeteroData()
data["transaction"].x = torch.from_numpy(X_scaled)
data["transaction"].y = torch.from_numpy(y.astype(np.int64))
data["transaction"].time = torch.from_numpy(df["TransactionDT"].values.astype(np.int64))
data["transaction"].train_mask = torch.from_numpy(is_train.values)
data["transaction"].test_mask  = torch.from_numpy((~is_train).values)

data["card"].num_nodes     = len(card_map)
data["merchant"].num_nodes = len(merch_map)
data["device"].num_nodes   = len(dev_map)

data["transaction", "uses_card",        "card"       ].edge_index = e_uses_card
data["transaction", "at_merchant",      "merchant"   ].edge_index = e_at_merchant
data["transaction", "on_device",        "device"     ].edge_index = e_on_device
data["transaction", "velocity_cluster", "transaction"].edge_index = e_velocity_cluster

print(data)

## 6. Validation checks

In [ ]:
n_tx = data["transaction"].num_nodes
assert data["transaction"].x.shape == (n_tx, X_scaled.shape[1])
assert data["transaction"].train_mask.sum() + data["transaction"].test_mask.sum() == n_tx
# temporal masks must be a time-prefix (nodes are time-sorted)
n_tr = int(data["transaction"].train_mask.sum())
assert bool(data["transaction"].train_mask[:n_tr].all()), "train mask is not a time prefix"
assert data["transaction", "uses_card", "card"].edge_index[1].max() < data["card"].num_nodes
assert data["transaction", "at_merchant", "merchant"].edge_index[1].min() >= 0
assert data["transaction", "at_merchant", "merchant"].edge_index[1].max() < data["merchant"].num_nodes
assert data["transaction", "on_device", "device"].edge_index[1].max() < data["device"].num_nodes
assert data["transaction", "velocity_cluster", "transaction"].edge_index.max() < n_tx
# velocity edges must be causal: src earlier than (or equal to) dst
t = data["transaction"].time
vc = data["transaction", "velocity_cluster", "transaction"].edge_index
assert bool((t[vc[0]] <= t[vc[1]]).all()), "velocity_cluster edge points backward in time"

print(f"Feature dim: {data['transaction'].x.shape[1]}")
print(f"Train fraud: {y[is_train.values].mean():.3%}   Test fraud: {y[(~is_train).values].mean():.3%}")
print("All structural checks passed.")

## 7. Save artifacts

In [ ]:
torch.save(data, f"{OUT}/graph.pt")

graph_meta = {
    "num_transactions": n_tx,
    "num_cards": len(card_map),
    "num_merchants": len(merch_map),
    "num_devices": len(dev_map),
    "feature_cols": feat_cols,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "cat_maps": cat_maps,
    "cat_normalization": "code+1 / cardinality+1 (missing -> 0)",
    "medians": med.to_dict(),
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "card_map": {float(k): int(v) for k, v in card_map.items()},
    "merch_map": {str(k): int(v) for k, v in merch_map.items()},
    "dev_map": {str(k): int(v) for k, v in dev_map.items()},
    "merchant_requires_addr1": True,
    "split_cutoff": int(cutoff),
    "device_unk": DEVICE_UNK,
    "velocity_window_sec": WIN,
}
with open(f"{OUT}/graph_meta.pkl", "wb") as f:
    pickle.dump(graph_meta, f)

print("Saved: graph.pt, graph_meta.pkl")
print("Upload BOTH to the fraudgraph-stage2 Kaggle dataset (new version) so notebooks 03/04 pick them up.")